In [1]:
import os
import re
import json

cleaned_docs_path = os.path.abspath("../documents/cleaned")

In [2]:
nutrition_test = "Figure 2A shows the daily energy expenditure differences between isocaloric diets with equal protein but differing in the ratio of carbohydrate to fat. The pooled weighted mean difference in energy expenditure was 26 kcal/d (P <.0001) greater with lower fat diets. Figure 2B shows differences in the rate of body fat change between diets with the pooled weighted mean difference of 16 g/d (P <.0001) greater body fat loss in favor of the lower fat diets. These results are in the opposite direction to the predictions of the carbohydrateinsulin model, but the effect sizes are so small as to be physiologically meaningless. In other words, for all practical purposes “a calorie is a calorie” when it comes to body fat and energy expenditure differences between controlled isocaloric diets varying in the ratio of carbohydrate to fat. Nevertheless, it is possible that isocaloric diets differing in carbohydrate and fat may have overall health effects unrelated to total body fat or energy expenditure.58 For example, dietary carbohydrates may play an important role in determining the anatomic location of body fat stores, with diets higher in refined carbohydrate possibly leading to increased visceral and liver fat.59,60 Diet composition may also influence energy intake when the amount of the diet consumed is not controlled. For example, increasing dietary fat results in greater energy intake61,62 and decreasing dietary fat has the opposite effect.63,64 However, very low carbohydrate, high-fat diets may reduce appetite by promoting an increase in circulating ketones,65 although the mechanism for this effect is unclear.66 Furthermore, low carbohydrate diets often increase protein intake, which may independently increase satiety and decrease overall energy intake.46,47 These effects may help explain the short-term benefits for weight loss of lower carbohydrate, higher protein diets.54–56 However, long-term weight loss diet studies targeting different macronutrients demonstrate similar mean body weight trajectories corresponding to a similar exponential decay of diet adherence with all diets.67 The reasons for the long-term erosion of diet adherence are not well understood. Likely factors include practical challenges in sustaining diet changes in the face of engrained food habits developed over years of living in an obesogenic Hall and Guo Gastroenterology."

workout_test = "Repetition Speed The speed with which a lifter performs repetitions can impact the hypertrophic response. Despite limitations both in the quantity of research and aspects of study design, some conclusions can be drawn on the topic. With respect to concentric repetitions, there is some evidence that faster repetitions are beneﬁcial for hypertrophy. Nogueira et al. (133) found that performing concentric actions at 1-second cadence instead of three seconds had greater impact on both upper and lower limb muscle thickness in elderly men. This may be attributed to an increased recruitment and corresponding fatigue of high-threshold MUs. Other studies, however, suggest that training at moderate speeds has greater effects on hypertrophy (56), perhaps through a heightened metabolic demand (12). Maintaining continuous muscle tension at moderate repetition speeds also has been shown to enhance muscle ischemia and hypoxia, thereby augmenting the hypertrophic response (1 74)."

## KeyWords Extraction

### spaCy

In [3]:
from collections import Counter
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_keywords(text, top_n=15, n=1):
    doc = nlp(text.lower())

    keywords = []

    # n = 1: single-word keywords
    if n == 1:
        for token in doc:
            if (
                token.is_alpha
                and not token.is_stop
                and not token.is_punct
                and token.pos_ in ["NOUN", "PROPN", "ADJ", "VERB"]
            ):
                keywords.append(token.lemma_)

    # n >= 2: multi-word keywords using noun chunks
    else:
        for chunk in doc.noun_chunks:
            words = []

            for token in chunk:
                if (
                    token.is_alpha
                    and not token.is_stop
                    and not token.is_punct
                    and token.pos_ in ["NOUN", "PROPN", "ADJ", "VERB"]
                ):
                    words.append(token.lemma_)

            if 1 <= len(words) <= n:
                keywords.append(" ".join(words))

    keyword_counts = Counter(keywords).most_common(top_n)

    return [keyword for keyword, count in keyword_counts]

In [4]:
keywords = extract_keywords(nutrition_test, top_n=5,n=2)
print("Nutrition Test Keywords:", keywords)

Nutrition Test Keywords: ['carbohydrate', 'diet', 'isocaloric diet', 'ratio', 'calorie']


In [5]:
keywords = extract_keywords(workout_test, top_n=5, n=2)
print("Workout Test Keywords:", keywords)

Workout Test Keywords: ['hypertrophic response', 'hypertrophy', 'speed', 'lifter', 'repetition']


### YAKE

In [6]:
import yake

kw_extractor = yake.KeywordExtractor(lan="en", n=2, top=5)

keywords_with_scores = kw_extractor.extract_keywords(nutrition_test)

nutrition_keywords = [keyword for keyword, score in keywords_with_scores]
workout_keywords = [keyword for keyword, score in kw_extractor.extract_keywords(workout_test)]

print("Nutrition Test YAKE Keywords:", nutrition_keywords)
print("Workout Test YAKE Keywords:", workout_keywords)

Nutrition Test YAKE Keywords: ['fat', 'body fat', 'energy expenditure', 'expenditure differences', 'diets']
Workout Test YAKE Keywords: ['lifter performs', 'hypertrophic response', 'performs repetitions', 'repetitions', 'response']


### TF-IDF

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_tfidf_keywords(documents, top_n=10, ngram_range=(1, 2)):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=ngram_range
    )

    tfidf_matrix = vectorizer.fit_transform(documents)
    feature_names = vectorizer.get_feature_names_out()

    results = []

    for row in tfidf_matrix:
        scores = row.toarray().flatten()
        top_indices = scores.argsort()[::-1][:top_n]
        keywords = [feature_names[i] for i in top_indices if scores[i] > 0]
        results.append(keywords)

    return results

In [8]:
documents = [nutrition_test, workout_test]

tfidf_keywords = extract_tfidf_keywords(
    documents,
    top_n=5,
    
    ngram_range=(1, 2)
)

print("Nutrition TF-IDF:", tfidf_keywords[0])
print("Workout TF-IDF:", tfidf_keywords[1])

Nutrition TF-IDF: ['fat', 'diets', 'carbohydrate', 'energy', 'body']
Workout TF-IDF: ['muscle', 'repetitions', 'moderate', 'hypertrophic response', 'hypertrophy']


### LLL-Based Extraction

In [9]:
import ollama
import ast
import re

STOP_WORDS = frozenset({
    "study", "research", "results", "analysis", "paper", "data",
    "method", "methods", "approach", "finding", "findings", "work",
    "based", "using", "used", "new", "also", "well",
})

_LIST_PATTERN = re.compile(r'"([^"]+)"|\'([^\']+)\'')


def extract_llm_keywords(text, model="mistral", top_n=10):
    if not text or not text.strip():
        raise ValueError("text must be a non-empty string")

    prompt = f"""
You are a keyword extraction system.

Extract the {top_n} most important keywords.

Rules:
- Return ONLY a valid Python list of strings, e.g. ["word1", "word2"]
- No explanation, numbering, or markdown.
- Do not include stop words or punctuation.
- Do not include generic words like "study", "research", "results", etc.
- Prefer scientific and domain-specific terms.
- Avoid duplicate or very similar keywords.

Text:
{text}

Output:
"""

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.8},
    )

    raw_output = response["message"]["content"].strip()
    raw_output = re.sub(r"^```[a-z]*\n?|```$", "", raw_output, flags=re.MULTILINE).strip()

    try:
        keywords = ast.literal_eval(raw_output)
        if not isinstance(keywords, list):
            raise ValueError
        keywords = [item for sublist in keywords for item in (sublist if isinstance(sublist, list) else [sublist])]
    except (ValueError, SyntaxError):
        matches = _LIST_PATTERN.findall(raw_output)
        keywords = [a or b for a, b in matches]

    seen = set()
    result = []
    for kw in keywords:
        kw = kw.lower().strip()
        if kw and kw not in STOP_WORDS and kw not in seen:
            seen.add(kw)
            result.append(kw)

    return result[:top_n]

In [10]:
nutrition_keywords = extract_llm_keywords(nutrition_test, top_n=5)

In [11]:
workout_keywords = extract_llm_keywords(workout_test, top_n=5)

In [12]:
print("Nutrition Test LLM Keywords:", nutrition_keywords)
print("Workout Test LLM Keywords:", workout_keywords)

Nutrition Test LLM Keywords: ['isocaloric diets', 'daily energy expenditure', 'lower fat diets', 'body fat loss', 'carbohydrate to fat ratio']
Workout Test LLM Keywords: ['repetition speed', 'hypertrophic response', 'concentric repetitions', 'muscle thickness', 'moderate speeds']


In [13]:
text = "imulation are molecularly transduced to downstream targets that shift muscle protein balance to favor synthesis over degradation. Several primary anabolic signaling pathways have been identiﬁed including Akt/mammalian target of rapamycin (mTOR), mitogen-activated protein kinase (MAPK), and calcium-(Ca 2+) dependent pathways. The following is an overview of each of these pathways. 2858 Journal of Strength and Conditioning Research the TM Mechanisms of Muscle Hypertrophy\n\n#\n\nAkt/Mammalian Target of Rapamycin Pathway The Akt/mTOR pathway is believed to act as a master network regulating skeletal muscle growth (18,77,181). Although the speciﬁc molecular mechanisms have not been fully elucidated, Akt is considered a molecular upstream nodal point that is both an effector of anabolic signaling a"

print("test : ", extract_llm_keywords(text, top_n=5))

test :  ['simulation', 'molecularly transduced', 'downstream targets', 'muscle protein balance', 'akt/mammalian target of rapamycin']


## Fixed-size

In [14]:
chunk_size = 800
overlap_size = 100


def chunk(text, chunk_size=chunk_size, overlap_size=overlap_size):

    chunks = []
    strat = 0
    end = chunk_size
    while strat < len(text):
        chunk = text[strat:end]
        chunks.append(chunk)
        strat += chunk_size - overlap_size
        end += chunk_size - overlap_size
    return chunks

In [15]:
chunk_path = os.path.abspath("../documents/chunks/fixed-size")

if not os.path.exists(chunk_path):
    os.makedirs(chunk_path)

for cat in os.listdir(cleaned_docs_path):
    cat_path = os.path.join(cleaned_docs_path, cat)
    output_cat_path = os.path.join(chunk_path, cat)

    if not os.path.exists(output_cat_path):
        os.makedirs(output_cat_path)

    for doc in os.listdir(cat_path):
        doc_path = os.path.join(cat_path, doc)

        with open(doc_path, "r", encoding="utf-8") as f:
            text = f.read()

        chunks = chunk(text)
        source = f"{cat}/{doc}"

        chunk_objects = []

        for i, chunk_text in enumerate(chunks, start=1):
            chunk_objects.append(
                {
                    "id": f"{cat}_{doc[:-3]}_{i}",
                    "source": f"{cat}/{doc}",
                    "chunk": chunk_text,
                    "keywords": extract_llm_keywords(chunk_text, top_n=5)
                }
            )

        with open(
            os.path.join(output_cat_path, f"{doc[:-3]}_chunks.json"),
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(chunk_objects, f, ensure_ascii=False, indent=2)

        print(f"Chunks created for {doc} in {cat}")

Chunks created for the_mechanisms_of_muscle_hypertrophy_and_their.40.md in workout
Chunks created for progression_models_in_resistance_training_for.26.md in workout
Chunks created for basics_of_strength_and_conditioning_manual.md in workout
Chunks created for mss-51-094.md in workout
Chunks created for nihms851543.md in nutrition
Chunks created for 12970_2017_Article_189.md in nutrition
Chunks created for ncomms13103.md in nutrition
Chunks created for 439.full.md in nutrition


## Sentences

In [16]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /home/ismail/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ismail/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [17]:
chunk_size_sent = 10
overlap_size_sent = 2


def chunk_per_sentence(
    text, chunk_size_sent=chunk_size_sent, overlap_size_sent=overlap_size_sent
):
    sentences = nltk.sent_tokenize(text, language="english")
    chunks = []
    start = 0
    end = chunk_size_sent
    while start < len(sentences):
        chunk = " ".join(sentences[start:end])
        chunks.append(chunk)
        start += chunk_size_sent - overlap_size_sent
        end += chunk_size_sent - overlap_size_sent
    return chunks

In [18]:
chunk_path = os.path.abspath("../documents/chunks/sentences")

if not os.path.exists(chunk_path):
    os.makedirs(chunk_path)

for cat in os.listdir(cleaned_docs_path):
    cat_path = os.path.join(cleaned_docs_path, cat)
    output_cat_path = os.path.join(chunk_path, cat)

    if not os.path.exists(output_cat_path):
        os.makedirs(output_cat_path)

    for doc in os.listdir(cat_path):
        doc_path = os.path.join(cat_path, doc)

        with open(doc_path, "r", encoding="utf-8") as f:
            text = f.read()

        chunks = chunk_per_sentence(text, chunk_size_sent, overlap_size_sent)
        source = f"{cat}/{doc}"

        chunk_objects = []

        for i, chunk_text in enumerate(chunks, start=1):
            chunk_objects.append(
                {
                    "id": f"{cat}_{doc[:-3]}_{i}",
                    "source": f"{cat}/{doc}",
                    "chunk": chunk_text,
                    "keywords": extract_llm_keywords(chunk_text, top_n=5)
                }
            )

        with open(
            os.path.join(output_cat_path, f"{doc[:-3]}_chunks.json"),
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(chunk_objects, f, ensure_ascii=False, indent=2)

        print(f"Chunks created for {doc} in {cat}")

Chunks created for the_mechanisms_of_muscle_hypertrophy_and_their.40.md in workout
Chunks created for progression_models_in_resistance_training_for.26.md in workout
Chunks created for basics_of_strength_and_conditioning_manual.md in workout
Chunks created for mss-51-094.md in workout
Chunks created for nihms851543.md in nutrition
Chunks created for 12970_2017_Article_189.md in nutrition
Chunks created for ncomms13103.md in nutrition
Chunks created for 439.full.md in nutrition


## Similarity

In [107]:
import os
import re
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [113]:
def split_paragraphs(text):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

def similarity_chunks(text, max_chars=1800, sim_threshold=0.25):
    paras = split_paragraphs(text)
    if not paras:
        return []

    vec = TfidfVectorizer(stop_words="english")
    X = vec.fit_transform(paras)
    sims = cosine_similarity(X[:-1], X[1:]) if len(paras) > 1 else []

    chunks = []
    cur = ""

    for i, p in enumerate(paras):
        boundary = False
        if i < len(paras) - 1:
            sim = sims[i][0]
            if sim < sim_threshold:
                boundary = True

        if len(cur) + len(p) + 2 <= max_chars:
            cur = (cur + "\n\n" + p).strip()
        else:
            if cur.strip() and cur.strip() != "#":
                chunks.append(cur.strip())
            cur = p

        if boundary:
            if cur.strip() and cur.strip() != "#":
                chunks.append(cur.strip())
            cur = ""

    if cur.strip() and cur.strip() != "#":
        chunks.append(cur.strip())

    return chunks


def build_chunk_objects(cat, doc, chunks):
    objs = []
    for i, chunk_text in enumerate(chunks, start=1):
        if not chunk_text or not chunk_text.strip() or chunk_text.strip() == "#":
            continue
        objs.append({
            "id": f"{cat}_{doc[:-3]}_{i}",
            "source": f"{cat}/{doc}",
            "chunk": chunk_text,
            "keywords": extract_llm_keywords(chunk_text, top_n=5),
        })
    return objs

In [114]:
def build_chunk_objects(cat, doc, chunks):
    objs = []
    for i, chunk_text in enumerate(chunks, start=1):
        if not chunk_text or not chunk_text.strip():
            continue
        objs.append({
            "id": f"{cat}_{doc[:-3]}_{i}",
            "source": f"{cat}/{doc}",
            "chunk": chunk_text,
            "keywords": extract_llm_keywords(chunk_text, top_n=5),
        })
    return objs

In [116]:
chunk_path = os.path.abspath("../documents/chunks/similarity")

if not os.path.exists(chunk_path):
    os.makedirs(chunk_path)

for cat in os.listdir(cleaned_docs_path):
    cat_path = os.path.join(cleaned_docs_path, cat)
    output_cat_path = os.path.join(chunk_path, cat)

    if not os.path.exists(output_cat_path):
        os.makedirs(output_cat_path)

    for doc in os.listdir(cat_path):
        doc_path = os.path.join(cat_path, doc)

        with open(doc_path, "r", encoding="utf-8") as f:
            text = f.read()

        chunks = similarity_chunks(text, max_chars=1800, sim_threshold=0.25)
        chunk_objects = build_chunk_objects(cat, doc, chunks)

        with open(
            os.path.join(output_cat_path, f"{doc[:-3]}_chunks.json"),
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(chunk_objects, f, ensure_ascii=False, indent=2)

        print(f"Similarity chunks created for {doc} in {cat}")

Similarity chunks created for the_mechanisms_of_muscle_hypertrophy_and_their.40.md in workout
Similarity chunks created for progression_models_in_resistance_training_for.26.md in workout
Similarity chunks created for basics_of_strength_and_conditioning_manual.md in workout
Similarity chunks created for mss-51-094.md in workout
Similarity chunks created for nihms851543.md in nutrition
Similarity chunks created for 12970_2017_Article_189.md in nutrition
Similarity chunks created for ncomms13103.md in nutrition
Similarity chunks created for 439.full.md in nutrition
